In [80]:
import pandas as pd
import yaml
from itertools import product
import re
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.postgresql import insert as pg_insert

FOLDER = "data/STORE5/"
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

def read_files(folder):
    with open(f"{folder}/books.yaml", "r", encoding="utf-8") as file:
        data_books = yaml.safe_load(file)
    df_books = pd.DataFrame(data_books)
    df_orders = pd.read_parquet(f"{folder}/orders.parquet")
    df_users = pd.read_csv(f"{folder}/users.csv")

    return df_books, df_orders, df_users

df_books, df_orders, df_users =  read_files(FOLDER)


In [81]:
def display_df_info(df) :
    display(df.shape)
    df.info()
    df.head()

for df in [df_orders, df_users, df_books]:
    display_df_info(df)

(9714, 7)

<class 'pandas.DataFrame'>
RangeIndex: 9714 entries, 0 to 9713
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          9714 non-null   int64
 1   user_id     9714 non-null   int64
 2   book_id     9714 non-null   int64
 3   quantity    9714 non-null   int32
 4   unit_price  9714 non-null   str  
 5   timestamp   9714 non-null   str  
 6   shipping    7349 non-null   str  
dtypes: int32(1), int64(3), str(3)
memory usage: 875.6 KB


(3158, 5)

<class 'pandas.DataFrame'>
RangeIndex: 3158 entries, 0 to 3157
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       3158 non-null   int64
 1   name     3158 non-null   str  
 2   address  3046 non-null   str  
 3   phone    3158 non-null   str  
 4   email    3158 non-null   str  
dtypes: int64(1), str(4)
memory usage: 433.3 KB


(750, 6)

<class 'pandas.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   :id         750 non-null    int64 
 1   :title      750 non-null    str   
 2   :author     750 non-null    str   
 3   :genre      750 non-null    str   
 4   :publisher  748 non-null    str   
 5   :year       750 non-null    object
dtypes: int64(1), object(1), str(4)
memory usage: 83.3+ KB


In [82]:
def clean_column_names(df):
    df.columns = df.columns.str.replace('[: ]', '', regex=True)

def inspect_duplicates(df):
    display(f"Nr. of duplicate ids: {df['id'].duplicated().sum()}")
    display(f"Nr. of duplicate rows: {df.duplicated(subset=df.columns.drop('id')).sum()}")

def inspect_missing_values(df):
    display(df.isna().sum())
    display(df[df.isna().any(axis=1)])

def clean_strings(df):
    for col in df.select_dtypes(include=["object", "str"]).columns:
        df[col] = df[col].map( lambda x: x.strip() or None if isinstance(x, str) else x )

In [83]:

def inspect_book_column_types(df):
    display(df["publisher"].map(type).value_counts())
    display(df["year"].map(type).value_counts())
    display(df[df["year"].map(lambda x: isinstance(x, str))])

def clean_year(df):
    df["year"] = df["year"].map(lambda x: x if isinstance(x, int) and x > 0 else None)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    display(df["year"].describe())
    display(f"Years in future: {(df['year'] > pd.Timestamp.now().year).sum()}")

def books_ingest(df):
    clean_column_names(df)
    inspect_duplicates(df)
    inspect_missing_values(df)
    inspect_book_column_types(df)
    clean_year(df)
    clean_strings(df)

books_ingest(df_books)



'Nr. of duplicate ids: 0'

'Nr. of duplicate rows: 0'

id           0
title        0
author       0
genre        0
publisher    2
year         0
dtype: int64

,id,title,author,genre,publisher,year
226,18925,The Pianist,Barney Will,Fiction in verse,NaN,1982
688,18895,Batman: Arkham City,"Emanuel Rippin, Herbert Schmeler",Tall tale,NaN,1986


publisher
<class 'str'>      748
<class 'float'>      2
Name: count, dtype: int64

year
<class 'int'>    742
<class 'str'>      8
Name: count, dtype: int64

,id,title,author,genre,publisher,year
66,19552,The Stars' Tennis Balls,Rep. Charley Hickle,Fantasy,University of Michigan Press,-1
318,19390,Bad,Leola Mills,Textbook,D. Appleton & Company,-1
567,19386,Into the Wild,Rep. Zena Hansen,Folklore,Cengage Learning,-1
580,19311,Sonic Unleashed,Richelle Weber,Fiction in verse,Academic Press,
670,18900,Doom II,"Marian Sauer, Edythe Green",Fiction in verse,Palgrave Macmillan,-
696,18990,Command & Conquer 3: Tiberium Wars,Lawanna Leannon II,Mystery,Secker & Warburg,-
731,19227,Kirby's Dream Land,Zane Bartell,Comic/Graphic Novel,Marshall Pickering,
740,19405,The Mermaids Singing,Leisha Luettgen,Fantasy,Felony & Mayhem Press,-


count          740.0
mean     1994.539189
std        28.029654
min           1830.0
25%           1984.0
50%           1998.0
75%           2013.0
max           2026.0
Name: year, dtype: Float64

'Years in future: 0'

In [84]:
def extract_book_authors(df):
    rows = []
    for _, row in df[["id", "author"]].iterrows():
        authors = row["author"].split(",")
        for author in authors:
            author = author.strip()
            if author:
                rows.append({ "book_id": row["id"], "author_name": author})
    return pd.DataFrame(rows).drop_duplicates()

def load_existing_authors():
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT author_id, author_name FROM public.authors"), conn)

def create_authors_table(df_book_authors):
    df_authors_db = load_existing_authors()
    df_authors = pd.DataFrame({"author_name": df_book_authors["author_name"].drop_duplicates()}).reset_index(drop=True)

    existing = df_authors_db.set_index("author_name")["author_id"]
    df_authors["author_id"] = df_authors["author_name"].map(existing)
    new_mask = df_authors["author_id"].isna()
    if not df_authors_db.empty: 
        next_id = int(df_authors_db["author_id"].max()) + 1 
    else:
        next_id = 1
    df_authors.loc[new_mask, "author_id"] = range(next_id, next_id + new_mask.sum())
    df_authors["author_id"] = df_authors["author_id"].astype("int64")
    df_authors_new = df_authors.loc[new_mask].copy()

    df_book_authors = df_book_authors.merge(df_authors, on="author_name", how="left")
    df_book_authors = df_book_authors[["book_id", "author_id"]].drop_duplicates()

    return df_authors_new, df_book_authors
    
def normalize_authors(df):
    df_book_authors = extract_book_authors(df)
    df_authors, df_book_authors = create_authors_table(df_book_authors)
    df = df.drop(columns=["author"])

    return df, df_authors, df_book_authors

def load_existing_genres():
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT genre_id, genre FROM public.genres"), conn)
    
def normalize_genres(df):
    df_genres_db = load_existing_genres()
    df_genres = ( df[["genre"]].drop_duplicates().sort_values("genre").reset_index(drop=True))

    existing = df_genres_db.set_index("genre")["genre_id"]
    df_genres["genre_id"] = df_genres["genre"].map(existing)
    new_mask = df_genres["genre_id"].isna()
    if not df_genres_db.empty:
        next_id = int(df_genres_db["genre_id"].max()) + 1
    else:
        next_id = 1
    df_genres.loc[new_mask, "genre_id"] = range(next_id, next_id + new_mask.sum())
    df_genres["genre_id"] = df_genres["genre_id"].astype("int64")
    df_genres_new = df_genres.loc[new_mask, ["genre_id", "genre"]].copy()

    df = df.merge(df_genres, on="genre", how="left")
    df = df.drop(columns=["genre"])

    return df, df_genres_new

def normalize_books(df):
    df, df_authors, df_book_authors = normalize_authors(df)
    df, df_genres = normalize_genres(df)

    display(df_book_authors.head(3))
    display(df_authors.head(3))
    display(df_genres.head(3))
    display(df.head(3))
    display(f"Missing Author ids: {df_book_authors["author_id"].isna().sum()}")
    display(f"Missing Genre ids: {df["genre_id"].isna().sum()}") 

    return df, df_authors, df_book_authors, df_genres

df_books, df_authors, df_book_authors, df_genres = normalize_books(df_books)


,book_id,author_id
0,19437,1273
1,19437,1274
2,19437,1275


,author_name,author_id
0,Myles Rohan Sr.,1273
1,Shirl Conn,1274
2,Rep. Charley Hickle,1275


,genre_id,genre


,id,title,publisher,year,genre_id
0,19437,The Grand Budapest Hotel,Carlton Publishing Group,1980,27
1,19267,All the King's Men,Harvill Press at Random House,1995,2
2,19428,Once Upon a Time in the West,Legend Books,2004,4


'Missing Author ids: 0'

'Missing Genre ids: 0'

In [85]:
def inspect_address_types(df):
    display(df["address"].map(type).value_counts())

def clean_phone_numbers(df):
    df["phone"] = df["phone"].str.replace(r"\D", "", regex=True)
    display(f"Nr. of wrong phone numbers: {(df['phone'].str.len() != 10).sum()}")

def users_ingest(df):
    clean_column_names(df)
    inspect_duplicates(df)
    inspect_missing_values(df)
    inspect_address_types(df)
    clean_phone_numbers(df)
    clean_strings(df)

users_ingest(df_users)

'Nr. of duplicate ids: 0'

'Nr. of duplicate rows: 0'

id           0
name         0
address    112
phone        0
email        0
dtype: int64

,id,name,address,phone,email
2,53589,Floy Kling,NaN,(724) 943-2841,kurtis@stehr.test
47,53377,Erick Graham DO,NaN,(755) 611-6594,erasmo.bailey@stanton.test
61,52540,Sen. Jeraldine Brakus,NaN,278-196-1793,donte@muller.test
90,52136,Arnold Wisozk,NaN,803 760 6650,leopoldo@macejkovic-ritchie.test
125,52658,Elizabeth Jones,NaN,924.081.2799,majorie@runolfsdottir-dooley.example
...,...,...,...,...,...
3068,52905,Mechelle Gutmann LLD,NaN,(140) 875-3665,ross@crona.test
3095,51891,Janella Cremin,NaN,(254) 801-1043,vernon_wolf@prohaska.test
3107,52170,Quinn Feil,NaN,(464) 210-9833,elina.botsford@koch-west.example
3118,51197,Liane Shields,NaN,622 819 4254,jefferson@reichert.example


address
<class 'str'>      3046
<class 'float'>     112
Name: count, dtype: int64

'Nr. of wrong phone numbers: 0'

In [86]:
def clean_shipping(df):
    display(df["shipping"].map(type).value_counts())
    for i, x in df["shipping"].items():
        if pd.isna(x) or (isinstance(x, str) and x.strip().upper() in ("", "NULL")):
            df.at[i, "shipping"] = None

def check_user_id_exists(df_orders,df_users):
    missing = (df_orders["user_id"].isin(df_users["id"]) == False)
    display(f"Nr. orders with non-existent users: {missing.sum()}")
    display(df_orders.loc[missing, ["user_id"]])

def check_book_id_exists(df_orders,df_books):
    missing = (df_orders["book_id"].isin(df_books["id"]) == False)
    display(f"Nr. orders with non-existent books: {missing.sum()}")
    display(df_orders.loc[missing, ["book_id"]])

def check_quantity_distribution(df):
    display(df["quantity"].describe())
    display(f"Negative quantity values: {(df["quantity"] <= 0).sum()}")
    display(f"Non natural quantity values: {(df["quantity"] % 1 != 0).sum()}")

def normalize_timestamp_strings(df):
    timestamp_copy = df["timestamp"]
    timestamp_copy = timestamp_copy.str.replace("A.M.", "AM").str.replace("P.M.", "PM")
    timestamp_copy = timestamp_copy.str.replace("a.m.", "am").str.replace("p.m.", "pm")
    timestamp_copy = timestamp_copy.str.replace(";", " ").str.replace(",", " ")

    return timestamp_copy

def generate_timestamp_formats():
    uncovered_formats = []

    for date_sep, month, year, time, order in product(
        ["-", "/", "."],
        ["%b", "%B"],
        ["%Y", "%y"],
        ["%H:%M", "%H:%M:%S", "%I:%M %p", "%I:%M:%S %p"],
        [0, 1],
    ):
        dates = [
            f"%d{date_sep}{month}{date_sep}{year}",
            f"{year}{date_sep}{month}{date_sep}%d",
            f"{month}{date_sep}%d{date_sep}{year}",
            f"{year}{date_sep}%d{date_sep}{month}",
        ]

        for date in dates:
            format = f"{time} {date}" if order == 0 else f"{date} {time}"
            uncovered_formats.append(format)

    for weekday, month, time in product(
        ["%a", "%A"],
        ["%b", "%B"],
        ["%H:%M", "%H:%M:%S"]
    ):
        uncovered_formats.append(f"{weekday} {month} %d {time} %Y")

    for fraction in ["", ".%f"]:
        uncovered_formats.append( f"%Y-%m-%dT%H:%M:%S{fraction}" )

    for date_sep, year, time, order in product(
            ["-", "/"],
            ["%Y", "%y"],
            ["%H:%M", "%H:%M:%S", "%I:%M %p", "%I:%M:%S %p"],
            [0, 1],
        ):
            dates = [
                f"%m{date_sep}%d{date_sep}{year}",
                f"{year}{date_sep}%m{date_sep}%d",
            ]
    
            for date in dates:
                format = f"{time} {date}" if order == 0 else f"{date} {time}"
                uncovered_formats.append(format)

    uncovered_formats.append("%d.%m.%Y %H:%M:%S")

    return uncovered_formats

def parse_timestamps(df, timestamp_copy, formats):
    df["timestamp"] = pd.Series( pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

    for format in formats:
        mask = df["timestamp"].isna()
        parsed = pd.to_datetime(timestamp_copy.loc[mask], format=format, errors="coerce", utc=True)
        df.loc[mask, "timestamp"] = parsed

    invalid = df["timestamp"].isna()

    print("Nr. invalid timestamps:", invalid.sum())

    display(pd.DataFrame({
        "id": df.loc[invalid, "id"],
        "original_timestamp": timestamp_copy.loc[invalid],
        "parsed_invalid_timestamp": df.loc[invalid, "timestamp"]
    }))

def clean_timestamps(df):
    original = df["timestamp"].copy()
    timestamp_copy = normalize_timestamp_strings(df)
    formats = generate_timestamp_formats()
    parse_timestamps(df, timestamp_copy, formats)
    df["timestamp"] = df["timestamp"].dt.floor("s")

def orders_ingest(df, df_users, df_books):
    clean_column_names(df)
    inspect_duplicates(df)
    inspect_missing_values(df)
    clean_shipping(df)
    check_user_id_exists(df,df_users)
    check_book_id_exists(df,df_books)
    check_quantity_distribution(df)
    clean_timestamps(df)
    clean_strings(df)

orders_ingest(df_orders, df_users, df_books)

'Nr. of duplicate ids: 0'

'Nr. of duplicate rows: 0'

id               0
user_id          0
book_id          0
quantity         0
unit_price       0
timestamp        0
shipping      2365
dtype: int64

,id,user_id,book_id,quantity,unit_price,timestamp,shipping
8,74169,53865,19148,1,EUR 46.5,"21-OCT-2025, 09:47:23 AM",NaN
12,71939,54320,19573,3,USD 31.00,"5-DEC-2025, 17:45",NaN
13,79098,53387,19442,5,32 $,13:37;20-Jan-2026,NaN
15,78304,52999,18871,1,51.0 USD,"10:16:45 pm, 2025-11-25",NaN
25,76650,51643,19299,1,56.99 USD,2025-03-29T03:28:24.490,NaN
...,...,...,...,...,...,...,...
9702,73792,51417,19156,1,EUR38.0,Thu Jun 26 23:46:59 2025,NaN
9705,72103,53395,18888,2,51.25 EUR,2025-09-13T18:54:31,NaN
9709,75666,51305,18910,1,€41.99,15:15:52 2026-01-26,NaN
9711,72466,54180,19249,2,45.5$,03/26/26 11:24:02 am,NaN


shipping
<class 'str'>      7349
<class 'float'>    2365
Name: count, dtype: int64

...

count    9714.000000
mean        1.527589
std         0.983604
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         5.000000
Name: quantity, dtype: float64

'Negative quantity values: 0'

'Non natural quantity values: 0'

Nr. invalid timestamps: 0


,id,original_timestamp,parsed_invalid_timestamp


In [87]:
def identify_currency(value):
    value = str(value).strip().upper()
    if "EUR" in value or "€" in value:
        currency = "EUR"
    elif "USD" in value or "$" in value:
        currency = "USD"
    else:
        currency = None
    value = value.replace("EUR", " ").replace("USD", " ").replace("€", " ").replace("$", " ").strip()

    return value, currency

def extract_cent_symbol(value):
    parts = re.findall(r"\d+", value)
    if len(parts) == 1:
        return "0." + parts[0]
    elif len(parts) == 2:
        return parts[0] + "." + parts[1]

    return None

def extract_decimal_symbol(value):
    parts = re.split(r"[.,]", value)
    if len(parts) == 1:
        return parts[0]
    elif len(parts) == 2 and parts[0] == "":
        if parts[1].isdigit() and len(parts[1]) <= 2:
            return "0." + parts[1]
    elif len(parts) == 2 and parts[0].isdigit():
        if parts[1] == "":
            return parts[0]
        elif parts[1].isdigit() and len(parts[1]) <= 2:
            return parts[0] + "." + parts[1]

    return None

def extract_price(value):
    if pd.isna(value):
        return None

    if "¢" in value:
        value = extract_cent_symbol(value)
    else:
        value = extract_decimal_symbol(value)

    if value is None or not value.replace(".", "", 1).isdigit():
        return None

    price = float(value)
    return round(price, 2) if price < 100 else None

def clean_prices(df):
    original_price = df["unit_price"].copy()
    parsed_price = df["unit_price"].map(identify_currency)
    price_text = parsed_price.map(lambda x: x[0])
    currency = parsed_price.map(lambda x: x[1])

    df["unit_price"] = price_text.map(extract_price)
    df.loc[currency == "EUR", "unit_price"] *= 1.2
    df["unit_price"] = df["unit_price"].round(2)

    df_copy = df[["unit_price"]].copy()
    df_copy["original_price"] = original_price
    df_copy["price_text"] = price_text
    df_copy["currency"] = currency

    return df_copy

df_check = clean_prices(df_orders)

In [88]:
def check_prices(df):
    display(df.head())
    display(df.isna().sum())
    display(df.loc[df["unit_price"].isna()])

check_prices(df_check)

,unit_price,original_price,price_text,currency
0,55.50,$55¢50,55¢50,USD
1,50.75,$ 50.75,50.75,USD
2,23.75,$23.75,23.75,USD
3,36.30,30.25 €,30.25,EUR
4,48.50,48.50USD,48.50,USD


unit_price        0
original_price    0
price_text        0
currency          0
dtype: int64

,unit_price,original_price,price_text,currency


In [89]:
def add_paid_price(df):
    df["paid_price"] = ( df["quantity"] * df["unit_price"]).round(2)
    display(df[["quantity", "unit_price", "paid_price"]].head())

def add_store_id(df_orders, df_users, df_books,df_book_authors):
    for df in [df_orders, df_users, df_books, df_book_authors]:
        store_id = int(FOLDER.split("STORE")[1].strip("/"))
        df["store_id"] = store_id

def prepare_for_postgresql():
    add_paid_price(df_orders)
    add_store_id(df_orders, df_users, df_books, df_book_authors)

    for df in [df_orders, df_users, df_books]:
        df.rename(columns={"id": "source_id"}, inplace=True)
        display(df.head()) 

prepare_for_postgresql()


,quantity,unit_price,paid_price
0,1,55.50,55.50
1,2,50.75,101.50
2,1,23.75,23.75
3,1,36.30,36.30
4,1,48.50,48.50


,source_id,user_id,book_id,quantity,unit_price,timestamp,shipping,paid_price,store_id
0,70365,51860,19516,1,55.50,2026-05-31 20:13:00+00:00,NaN,55.50,5
1,75931,52316,19014,2,50.75,2025-12-28 13:08:20+00:00,NaN,101.50,5
2,75546,54009,19457,1,23.75,2025-11-13 08:00:36+00:00,NaN,23.75,5
3,74431,53245,19161,1,36.30,2025-08-14 19:39:13+00:00,"Suite 587 7406 Ahmed Meadows, Spencershire, NM...",36.30,5
4,74204,51627,19108,1,48.50,2025-07-26 09:46:00+00:00,"100 Balistreri Camp, Auerfurt, MO 30489",48.50,5


,source_id,name,address,phone,email,store_id
0,53162,Arnoldo Moore,"Suite 617 27446 Tremblay Divide, Gislasonside,...",9887240521,guy.jacobson@schumm.example,5
1,52237,Kenisha Zemlak Sr.,"2634 Fahey Forks, East Vance, CO 54477",9391482312,marshall_johnston@homenick.example,5
2,53589,Floy Kling,NaN,7249432841,kurtis@stehr.test,5
3,54329,Desmond Adams,"Apt. 594 4933 Walsh Vista, West Brandenside, L...",3363840025,joannie.larson@gulgowski.test,5
4,53290,Torrie Sipes,"Apt. 442 4822 Sid Plaza, Dietrichhaven, TN 67794",8165223840,earl_moen@flatley.test,5


,source_id,title,publisher,year,genre_id,store_id
0,19437,The Grand Budapest Hotel,Carlton Publishing Group,1980,27,5
1,19267,All the King's Men,Harvill Press at Random House,1995,2,5
2,19428,Once Upon a Time in the West,Legend Books,2004,4,5
3,19183,A Swiftly Tilting Planet,NaN,1928,12,5
4,19491,WarioWare: Touched!,Eel Pie Publishing,2024,20,5


In [90]:
def insert_dataframes(table, conn, keys, data_iter):
    rows = [dict(zip(keys, row)) for row in data_iter]
    stmt = pg_insert(table.table).values(rows)
    stmt = stmt.on_conflict_do_nothing()
    result = conn.execute(stmt)
    return result.rowcount

tables = [
    ("authors", df_authors),
    ("genres", df_genres),
    ("books", df_books),
    ("book_authors", df_book_authors),
    ("users", df_users),
    ("orders", df_orders)
]

with engine.begin() as conn:
    for table, df in tables:
        inserted = df.to_sql(name=table, con=conn, schema="public", if_exists="append", index=False, method=insert_dataframes, chunksize=1000)
        print(f"{table}: {inserted} rows")

engine.dispose()


authors: -1 rows
genres: 0 rows
books: -1 rows
book_authors: -2 rows
users: -4 rows
orders: -10 rows
